# Gold reporting products

Build all eight facts, including budget and order events, and all seven Gold products. Coverage columns explain where source limitations restrict the measures.

Run from the `gold/` directory after setting `silver_manifest` in `gold_settings.json`.

In [1]:
from pathlib import Path

from IPython.display import display
import pandas as pd

from gold_pipeline import build, publish

SETTINGS_PATH = Path("gold_settings.json")
gold = build(SETTINGS_PATH, stage="products")
print("Silver release", gold.manifest["model_run_id"])
print("Cutoff", gold.settings["cutoff"])
display(pd.DataFrame([{"table": name, "rows": len(rows)} for name, rows in gold.tables.items()]))

Silver release 409a707f49d90b112334f8bb4bfd27423a7f3f48f7039eecdff871dddee256e5
Cutoff 2026-12-16


,table,rows
0,dim_legal_entity,2
1,dim_customer,5
2,dim_product,5
3,dim_date,14
4,dim_currency,1
5,fact_sales_order_booking,8
6,dim_warehouse,3
7,fact_shipment_line,5
8,dim_document_type,2
9,fact_billing_document,8


In [2]:
for name in ("fact_budget", "fact_order_event") + tuple(
    name for name in gold.tables if name.startswith("gold_")
):
    print(name)
    display(pd.DataFrame(gold.records(name)).head(20))

fact_budget


,id,legal_entity_id,period,fiscal_period,scenario,budget_version,scope_type,scope,currency,budget_amount,fiscal_period_key,legal_entity_key,currency_key,scenario_key,budget_version_key,planning_scope_key
0,"[""parent"",""product_category"",""PUMP"",""2026-11"",...",parent,2026-11,None,plan,files/otc-inbound/parent/finance/dealer-order-...,product_category,PUMP,INR,120000.000000000000,fiscal_period:__unknown__,legal_entity:parent,currency:INR,budget_scenario:plan,budget_version:files/otc-inbound/parent/financ...,"planning_scope:[""parent"",""product_category"",""P..."
1,"[""dealer"",""local_account"",""400-D"",""2026-11"",""I...",dealer,2026-11,None,plan,files/otc-inbound/dealer/finance/dealer-order-...,local_account,400-D,INR,50000.000000000000,fiscal_period:__unknown__,legal_entity:dealer,currency:INR,budget_scenario:plan,budget_version:files/otc-inbound/dealer/financ...,"planning_scope:[""dealer"",""local_account"",""400-D""]"


fact_order_event


,id,legal_entity_id,logical_event_id,scoped_order_id,event_timestamp_utc,event_type,channel,event_count,event_date_key,event_time_key,event_type_key,channel_key,legal_entity_key
0,"[""parent"",""P-E1""]",parent,P-E1,"[""parent"",""SAL-ORD-2026-00001""]",2026-11-21 11:26:42.026905+00:00,accepted,None,1,date:2026-11-21,time:11:26:42,order_event_type:accepted,channel:__unknown__,legal_entity:parent
1,"[""parent"",""P-BRONZE-R2-EVENT""]",parent,P-BRONZE-R2-EVENT,"[""parent"",""SAL-ORD-2026-00006""]",2026-11-21 15:25:39.819020+00:00,accepted,None,1,date:2026-11-21,time:15:25:39,order_event_type:accepted,channel:__unknown__,legal_entity:parent
2,"[""dealer"",""D-BRONZE-R2-EVENT""]",dealer,D-BRONZE-R2-EVENT,"[""dealer"",""D-BRONZE-R2-ORDER""]",2026-11-15 03:30:00+00:00,accepted,None,1,date:2026-11-15,time:03:30:00,order_event_type:accepted,channel:__unknown__,legal_entity:dealer
3,"[""dealer"",""D-E1""]",dealer,D-E1,"[""dealer"",""D-S1""]",2026-11-05 03:30:00+00:00,accepted,None,1,date:2026-11-05,time:03:30:00,order_event_type:accepted,channel:__unknown__,legal_entity:dealer


gold_sales_daily


,id,day,legal_entity_id,currency,booked_amount,net_billed_amount,distinct_orders,gross_margin,coverage
0,"[""2026-11-02"",""parent"",""INR""]",2026-11-02,parent,INR,100000.000000000000,0,1,None,"entity totals; region, category and margin def..."
1,"[""2026-11-03"",""parent"",""INR""]",2026-11-03,parent,INR,0,60000.000000000000,0,None,"entity totals; region, category and margin def..."
2,"[""2026-11-05"",""dealer"",""INR""]",2026-11-05,dealer,INR,40000.000000000000,40000.000000000000,1,None,"entity totals; region, category and margin def..."
3,"[""2026-11-05"",""parent"",""INR""]",2026-11-05,parent,INR,0,40000.000000000000,0,None,"entity totals; region, category and margin def..."
4,"[""2026-11-06"",""dealer"",""INR""]",2026-11-06,dealer,INR,40000.000000000000,0,4,None,"entity totals; region, category and margin def..."
5,"[""2026-11-15"",""dealer"",""INR""]",2026-11-15,dealer,INR,20000.000000000000,10000.000000000000,1,None,"entity totals; region, category and margin def..."
6,"[""2026-11-15"",""parent"",""INR""]",2026-11-15,parent,INR,20000.000000000000,5000.000000000000,1,None,"entity totals; region, category and margin def..."


... (10 outputs left out)

gold_collections_monthly


,id,due_month,legal_entity_id,currency,as_of_date,invoices,gross_due_amount,eligible_due_amount,cash_by_payment_posting_date,collection_rate,avg_days_to_pay,coverage
0,"[""2026-11"",""parent"",""INR"",""2026-12-16""]",2026-11,parent,INR,2026-12-16,1,20000.000000000000,10000.000000000000,0,None,None,application timestamps unavailable
1,"[""2026-12"",""dealer"",""INR"",""2026-12-16""]",2026-12,dealer,INR,2026-12-16,2,60000.000000000000,None,10000.000000000000,None,None,due date or credit linkage unresolved
2,"[""2026-12"",""parent"",""INR"",""2026-12-16""]",2026-12,parent,INR,2026-12-16,2,100000.000000000000,95000.000000000000,25000.000000000000,None,None,application timestamps unavailable


gold_pnl_vs_budget


,id,legal_entity_id,period,fiscal_period,scope_type,scope,scenario,budget_version,currency,budget_amount,actual_amount,variance,coverage,as_of_date
0,"[""parent"",""product_category"",""PUMP"",""2026-11"",...",parent,2026-11,None,product_category,PUMP,plan,files/otc-inbound/parent/finance/dealer-order-...,INR,120000.000000000000,None,None,actual scope unavailable or fiscal calendar un...,2026-12-16
1,"[""dealer"",""local_account"",""400-D"",""2026-11"",""I...",dealer,2026-11,None,local_account,400-D,plan,files/otc-inbound/dealer/finance/dealer-order-...,INR,50000.000000000000,None,None,actual scope unavailable or fiscal calendar un...,2026-12-16


In [3]:
display(pd.DataFrame(gold.checks))
display(pd.DataFrame(gold.exceptions))

,rule,expected,actual,passed
0,bookings_preserve_eligible_line_count,8,8,True
1,bookings_preserve_eligible_line_amount,220000.000000000000,220000.000000000000,True
2,baseline_booking_rows,2,2,True
3,baseline_distinct_orders,2,2,True
4,baseline_booking_inr,140000,140000.000000000000,True
5,baseline_shared_customer_product,1,1,True
6,baseline_pumps_each,11,11.000000000000,True
7,baseline_net_billing_inr,135000,135000.000000000000,True
8,baseline_gl_net_revenue_inr,135000,135000.000000000000,True
9,baseline_reconciliation_is_matched,True,True,True


,rule,source_id
0,booking_not_eligible,"[""parent"",""SAL-ORD-2026-00002"",""vqk0j2tpqq""]"
1,booking_not_eligible,"[""parent"",""SAL-ORD-2026-00003"",""vqltad6jqq""]"
2,booking_not_eligible,"[""parent"",""SAL-ORD-2026-00004"",""vqlfquq375""]"
3,booking_not_eligible,"[""parent"",""SAL-ORD-2026-00005"",""vqm83vc86c""]"
